# 01 - Data Quality

Load market data, enforce **point-in-time (PIT)** discipline on fundamentals, and scan engineered features for **look-ahead leakage**. These are the first two of quantcortex's seven enforced backtesting pitfalls.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.data.local_csv import load_ohlcv_csv, load_price_matrix

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
OHLCV_CSV = os.environ.get("QUANTCORTEX_OHLCV_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill(limit=5).dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


def load_ohlcv(symbol, start="2018-01-01"):
    """Load one symbol's real OHLCV data for feature engineering."""
    if OHLCV_CSV is not None:
        return load_ohlcv_csv(OHLCV_CSV, start=start)
    if not LIVE_YFINANCE:
        raise RuntimeError(
            "Set QUANTCORTEX_OHLCV_CSV=/path/to/ohlcv.csv for the Alpha158 cell."
        )

    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    frame = YFinanceProvider().fetch_ohlcv([symbol], start=start).get(symbol)
    if frame is None or frame.empty:
        raise RuntimeError(f"yfinance returned no OHLCV data for {symbol}")
    return frame.dropna()


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


## Load prices
This notebook uses only the explicitly selected real-data source.


In [ ]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM"]
prices = load_prices(symbols)
returns = prices.pct_change(fill_method=None).dropna()
print("prices:", prices.shape)
prices.tail()

## Point-in-time enforcement
Fundamentals must be keyed on **announcement_date**, never period_end. `PITEnforcer` raises if any feature would be known before it was announced.

In [ ]:
from quantcortex.data.processors.pit_enforcer import PITEnforcer, PITViolationError

fund = pd.DataFrame({
    "symbol": ["AAPL", "AAPL", "MSFT"],
    "period_end": pd.to_datetime(["2023-03-31", "2023-06-30", "2023-03-31"]),
    "announcement_date": pd.to_datetime(["2023-05-04", "2023-08-03", "2023-04-25"]),
    "feature_date": pd.to_datetime(["2023-05-05", "2023-08-04", "2023-04-26"]),
    "field": ["eps", "eps", "eps"],
    "value": [1.52, 1.26, 2.45],
})
clean = PITEnforcer().enforce(fund)
print("PIT enforcement passed:", len(clean), "rows clean")

In [ ]:
# A leaking row: feature_date BEFORE the announcement -> must raise.
bad = fund.copy()
bad.loc[0, "feature_date"] = pd.Timestamp("2023-04-01")  # before the 2023-05-04 announcement
try:
    PITEnforcer().enforce(bad)
    print("ERROR: violation not caught!")
except PITViolationError as e:
    print("Correctly raised PITViolationError:", str(e)[:120])

## Look-ahead leakage scan
`LookaheadDetector` flags a feature built from *future* data (`close.shift(-1)`) while clearing strictly-causal features.

In [ ]:
from quantcortex.data.processors.lookahead_detector import LookaheadDetector

close = prices.iloc[:, 0]
feat = pd.DataFrame(index=close.index)
feat["causal_ma"] = close.rolling(10).mean().shift(1)   # only past data
feat["causal_ret"] = close.pct_change(fill_method=None).shift(1)
feat["LEAKY_future"] = close.shift(-1)                  # tomorrow used today

report = LookaheadDetector().scan(feat, reference=close)
print(report.summary())
assert "LEAKY_future" in report.flagged_columns
assert "causal_ma" not in report.flagged_columns
print("\nFlagged:", report.flagged_columns)

## Data-quality visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(prices / prices.iloc[0]).plot(ax=ax[0], lw=1)
ax[0].set_title("Normalised prices"); ax[0].set_ylabel("growth of $1")
returns.iloc[:, 0].hist(bins=60, ax=ax[1])
ax[1].set_title(f"{symbols[0]} daily return distribution")
plt.tight_layout()
miss = prices.isna().mean().mean()
print(f"missing data fraction: {miss:.4%}")
print("data quality checks complete.")